# AstraFlame — Model Selection v2 + v3 (Google Colab)

**NASA Space Apps Challenge 2026 · Challenge: *Flame in Freefall* · Team Agnishikha**
Notebook by **Prachya Biswas**, Data Science Lead

This notebook rebuilds the five AstraFlame fire-safety models from NASA International Space Station (ISS) combustion data and shows how each model was chosen.

| # | Model | Question it answers | NASA data |
|---|---|---|---|
| 1 | `bass2_flame_spread` | How fast does flame spread over a PMMA rod in opposed airflow? | BASS-II (PSI-25) |
| 2 | `flex_extinction` | Will a fuel-droplet flame go out before the fuel is used up? | FLEX (PSI-69) |
| 3 | `flex_burning_rate` | How fast does the droplet burn (constant K)? | FLEX (PSI-69) |
| 4 | `spice_smoke_point` | How sooty is a gas-jet flame (smoke-point length)? | SPICE (PSI-107) |
| 5 | `same_particle_size` | How large are smoke particles from an overheated material? | SAME / SAME-R (PSI-101/102) |

**How to run:** `Runtime → Run all`. With the default fast mode (`RUN_MODE = "reuse"`), the whole notebook takes about 30 minutes. A full retrain of all five models takes about 50 minutes on Colab's two CPU cores.

**Contents**
1. Setup
2. The NASA data
3. Why grouped validation matters (leakage demo)
4. How good *can* a model get? (noise ceiling)
5. Nested model selection, step by step (Model 1)
6. Train all five models (v2), then 6b: the objective-driven v3 model (PI-GP)
7. Results: v1 vs v2
8. Physics checks
9. Try the models
10. Limits, data sources, use of AI

## 1. Setup

The models were built with **scikit-learn 1.9.1**, so this cell installs that version first. If it is not available for this Colab Python, the cell installs the newest version instead, and step 6 then retrains from scratch.

The code is cloned from the public team repository, so no login is needed.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn==1.9.1"], capture_output=True, text=True)
if r.returncode != 0:
    print("scikit-learn 1.9.1 not available here -> installing the latest version")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "scikit-learn>=1.6"], check=True)

!git clone -q https://github.com/prachyabiswas/NASA-Space-Apps-2026.git 2>/dev/null || (cd NASA-Space-Apps-2026 && git pull -q)
%cd /content/NASA-Space-Apps-2026/models

import sklearn, numpy as np, pandas as pd, warnings, json, os
import config as C            # every data / artifact / report / figure path
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__, "| CPU cores:", os.cpu_count())

## 2. The NASA data

Every row below is one measurement from an ISS experiment, and every value is traceable to a NASA Physical Sciences Informatics (PSI) record (CC0 licence).

The **group** column matters: rows from the same ISS test, session or run share hardware and conditions. Those rows are always kept together when the data is split.

In [ ]:
from ignis_models import bass2_features, flex2_features, spice_features, same2_features

bass = pd.read_csv(C.data("BASS-II_flame_spread_microgravity_v1.0.csv"))     # data/processed is the single source of truth
flex = pd.read_csv(C.data("FLEX_PSI-69_clean_v1.0.csv"))
spice = pd.read_csv(C.data("SPICE_PSI-107_smoke_points_v1.0.csv"))
same = pd.read_csv(C.data("SAME_PSI-101-102_smoke_particles_v1.0.csv"))

overview = pd.DataFrame([
    ["1 BASS-II flame spread", len(bass), bass.test_id.nunique(), "ISS test", "flame_spread_mm_s"],
    ["2/3 FLEX droplets", len(flex), flex.flex_id.str[:3].nunique(), "FLEX test series", "extinguished / burning_rate_K_mm2_s"],
    ["4 SPICE smoke point", len(spice), spice.gmt_day_2009.nunique(), "ISS session (day)", "smoke_point_length_mm"],
    ["5 SAME smoke particles", len(same), same.same_run.str.rstrip("a").nunique(), "run (fresh + aged reading)", "count_mean_diameter_um"],
], columns=["dataset", "rows", "groups", "group = ", "target"])
display(overview)
display(bass[["test_id", "o2_pct", "opposed_flow_cm_s", "diameter_cm", "flame_spread_mm_s", "source"]].head())

### Is the data trustworthy? (reproducibility + data contract)

Two checks run before any model is trained:
- **Reproducibility.** `etl` rebuilds every processed file from the verbatim NASA extracts and proves it is byte-for-byte identical. This includes 890 value checks against the NASA files.
- **Data contract.** `validate` checks schema, keys, row counts, categories, physics bounds, cross-field consistency and lineage.

In [ ]:
!python ../run_pipeline.py --only etl validate

## 3. Why grouped validation matters

Suppose rows from one ISS test land in both the training set and the test set. The model has then already seen that test, so the score is too good.

Below, the same model is scored two ways on BASS-II:
- **Random split:** leaks rows of the same test across training and testing.
- **Leave-one-test-out:** the honest way. The model never sees the test it is scored on.

In [ ]:
from sklearn.model_selection import KFold, LeaveOneGroupOut, cross_val_predict
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import r2_score

Xb, yb, gb = bass2_features(bass), bass.flame_spread_mm_s.values, bass.test_id.values
et = ExtraTreesRegressor(300, min_samples_leaf=1, random_state=0)
r2_random = r2_score(yb, cross_val_predict(et, Xb, yb, cv=KFold(5, shuffle=True, random_state=0)))
r2_group = r2_score(yb, cross_val_predict(et, Xb, yb, groups=gb, cv=LeaveOneGroupOut()))
print(f"Random 5-fold split (leaky)  R² = {r2_random:.2f}")
print(f"Leave-one-ISS-test-out       R² = {r2_group:.2f}   <- the honest number")

## 4. How good *can* a model get? (noise ceiling)

Sometimes two different ISS tests ran under almost identical conditions. Their results should match, and the gap between them is run-to-run noise that no model can explain.

From these near-replicate pairs:

$\text{noise variance} \approx \tfrac{1}{2}\,\overline{(y_i-y_j)^2}$, and $\;R^2_{\max} \approx 1 - \dfrac{\text{noise variance}}{\text{total variance}}$

This is approximate, because near-replicates are not perfect replicates.

In [ ]:
!python noise_ceiling.py

## 5. Nested model selection, step by step (Model 1, BASS-II)

**The mistake v2 fixes.** If you pick the best model by its cross-validation score and then report that same score, the number is too optimistic. It already helped choose the winner.

**Nested cross-validation** keeps the two jobs apart:

```
OUTER loop: hold out one whole ISS test                      -> honest score
    INNER loop (training tests only):
        tune 8 model families (Ridge, interactions, kNN, SVR,
        Gaussian process, random forest, extra trees, boosting)
        apply a selection procedure -> one model
    predict the held-out test
```

Three selection procedures are compared here:
- best inner score
- 1-SE rule (the simplest model that is nearly as good)
- top-3 average

A fourth procedure, top-3 average with a physics-form fallback outside the tested domain, is added in step 6, because it needs the domain check.

This cell runs the full nested loop for BASS-II, which takes about 8 minutes.

In [ ]:
import time
from ml_selection import nested_cv, summarise
t0 = time.time()
res_b = nested_cv(Xb, yb, gb, "regression", outer="logo", transform="tune", verbose=False)
tab_b = summarise(res_b, yb, gb, "regression")
print(f"done in {time.time()-t0:.0f} s  (19 outer folds x 8 tuned families)")
display(tab_b[["model", "MAE", "RMSE", "R2", "picked_as_best_in_pct_of_outer_folds"]].round(4))

**How to read the table**
- The **family** rows show each family's score when it is tuned inside the inner loop and scored on held-out tests.
- The **[procedure]** rows show the honest score of each complete selection rule.
- The **picked as best** column shows how often each family won inside the inner loop. When the winner changes from fold to fold, picking a single model is noisy. Averaging the top-3 removes that noise.

## 6. Train all five models

- `RUN_MODE = "reuse"` loads the nested-CV results saved in the repository and then refits the final models. It takes about 3 minutes and gives exactly the scores in the README.
- `RUN_MODE = "full"` reruns everything from scratch. It takes about 40 minutes.

If the saved results cannot be loaded (for example, a different scikit-learn version), the script switches to the full run by itself.

The script picks **one** procedure for all five models: the one with the best mean rank across the five datasets. Choosing a different procedure for each model would let selection bias back in.

In [ ]:
RUN_MODE = "reuse"   # "reuse" (about 3 min) or "full" (about 40 min)
FLAG = "--reuse" if RUN_MODE == "reuse" else ""
!python train_v2.py {FLAG} 2>&1 | grep -v "^  repeat"

## 6b. v3: choose the model from the objective (physics-informed Gaussian process)

**Objective:** a fire-risk envelope. For given cabin conditions: will a flame survive, how fast does it spread or burn, and how sure is the answer.

The three outputs that feed the safe-cabin map (survive?, spread, burning rate) must meet three conditions:
- be accurate on unseen ISS tests
- follow physics
- give uncertainty that grows outside NASA's data

The model built for that is the **PI-GP**: a physics law, plus a GP correction kept only if grouped inner CV supports it, plus an ISS-test offset. Its preprocessing (`preprocess_v3.py`) is designed for it: physics space, logged cleaning, test groups, and a measured noise prior.

The next cell scores the PI-GP against every tuned v2 family on the **same** held-out splits. It measures accuracy, 90 % coverage, and a physics check that moves O2, airflow or aging time across the tested range on every NASA row. It takes about 7 minutes. The cell after it builds the final v3 models, the safe-cabin map, the moving-average test and the scaling test, which takes about 5 minutes.

In [ ]:
!python compare_v3.py 2>&1 | tail -3
v3 = pd.read_csv(C.report("v3_comparison.csv"))
keep = ["PI-GP (proposed)", "Physics law only (PI-GP mean)", "v2: top-3 average + fallback", "Random forest (tuned, v2)", "Extra trees (tuned, v2)"]
display(v3[v3.model.isin(keep)].round(3))
pc = pd.read_csv(C.report("v3_physics_check.csv"))
display(pc.pivot(index="model", columns="model_task", values="rows_violating_pct"))

In [ ]:
!python v3_final.py {"--no-scaling" if RUN_MODE == "reuse" else ""} 2>&1 | grep -v Warning | tail -12
print(json.dumps(json.loads(C.report("preprocessing_log_v3.json").read_text())["flex_extinction"], indent=1))
from IPython.display import Image, display
display(Image(str(C.figure("fig_v3_safe_map_next_experiment.png")), width=1000))
display(pd.read_csv(C.report("v3_moving_average_test.csv")))

## 7. Results: v1 vs v2 on the same held-out tests

- **v1** is the earlier model, re-scored on the *same* outer splits, so the comparison is fair.
- **v2** is the honest nested score of the whole selection procedure.

In [ ]:
cmp = pd.read_csv(C.report("v1_vs_v2.csv"))
display(cmp)
summary = json.loads(C.report("model_summary_v2.json").read_text())
rows = []
for name, v in summary.items():
    if name.startswith("_"):
        continue
    val = v["validation"]
    main = ("R2", val.get("R2"), val.get("R2_95CI")) if "R2" in val else ("ROC_AUC", val.get("ROC_AUC"), val.get("ROC_AUC_95CI"))
    rows.append([name, v["chosen"], main[0], main[1], main[2], val.get("fallback_share_in_cv_pct", "-")])
display(pd.DataFrame(rows, columns=["model", "final model", "metric", "nested score", "95% CI (group bootstrap)", "% rows on physics fallback in CV"]))
print("Selection procedure used for all models:", summary["_selection"]["procedure"])

In [ ]:
!python make_figures_v2.py
from IPython.display import Image, display
display(Image(str(C.figure("fig_v1_vs_v2.png")), width=1000))
display(Image(str(C.figure("fig_v2_pred_vs_measured.png")), width=1000))

## 8. Physics checks

A model can score well and still be physically wrong. These checks confirm that the predictions move in the direction combustion physics expects.

In [ ]:
from ignis_models import predict
P = lambda name, **kw: predict(name, explain=False, **kw)

o2 = [13, 15, 18, 21, 25]
pe = [P("flex_extinction", fuel="Methanol", pressure_kpa=101.3, o2_pct=o, co2_pct=0, he_pct=0, d0_mm=3.0)["probability"] for o in o2]
k = [P("flex_burning_rate", fuel="Methanol", pressure_kpa=101.3, o2_pct=o, co2_pct=0, he_pct=0, d0_mm=3.0)["prediction"] for o in o2]
sp = {f: P("spice_smoke_point", fuel=f, burner_diameter_mm=1.6002, coflow_air_velocity_cm_s=30)["prediction"] for f in ["Ethylene", "Propane", "Propylene/N2 50%", "Propylene"]}
sa = [P("same_particle_size", material="Kapton", sample_temperature_c=420, air_velocity_cm_s=v, aging_time_s=t, campaign="SAME-R")["prediction"] for v, t in [(8.1, 35), (8.1, 560), (0, 35)]]
fs = [P("bass2_flame_spread", o2_pct=o, opposed_flow_cm_s=2.0, diameter_cm=0.64)["prediction"] for o in [16.8, 17.5, 18.5]]

checks = [
    ("FLEX: more O2 -> flame less likely to go out", pe, all(np.diff(pe) <= 1e-9)),
    ("FLEX: more O2 -> faster burning (K up)", k, all(np.diff(k) > 0)),
    ("SPICE: propylene sootiest; N2 dilution less sooty", sp, sp["Propylene"] < sp["Propylene/N2 50%"] < sp["Ethylene"]),
    ("SAME: aged and no-flow smoke has larger particles", sa, sa[1] > sa[0] and sa[2] > sa[0]),
    ("BASS-II: more O2 -> faster flame spread", fs, all(np.diff(fs) > 0)),
]
for text, values, ok in checks:
    vals = {kk: round(vv, 3) for kk, vv in values.items()} if isinstance(values, dict) else [round(v, 3) for v in values]
    print(("PASS " if ok else "FAIL ") + text + "  ", vals)

## 9. Try the models

Change the inputs and run the cell. Every answer includes:
- the prediction with a 90 % band
- a domain check (`in_domain`, `warnings`); outside the tested NASA range, the physics-form model answers
- exact Shapley contributions, where `base_value + sum(contributions) = prediction`
- the NASA source and the validation scores

In [ ]:
r = predict("bass2_flame_spread", o2_pct=17.5, opposed_flow_cm_s=1.67, diameter_cm=0.64)
for key in ["prediction", "unit", "interval_90", "in_domain", "confidence", "model_used", "warnings", "base_value", "contributions", "source"]:
    print(f"{key:14s}: {r[key]}")
print("check:", round(r["base_value"] + sum(r["contributions"].values()), 4), "=", r["prediction"])

In [ ]:
# outside the tested range -> flagged, and the physics-form model answers
r = predict("flex_extinction", fuel="Methanol", pressure_kpa=60, o2_pct=30, co2_pct=0, he_pct=0, d0_mm=6)
print(r["probability"], r["confidence"], "|", r["model_used"])
for w in r["warnings"]:
    print(" -", w)

## 10. Limits, data sources and use of AI

**Limits**
- The datasets are small: 52–256 rows per model. The 95 % confidence intervals are wide, so the scores should be read together with them.
- Flame spread and burning rate are close to their noise ceilings. More ISS experiments would help them more than more modelling.
- SAME combines two campaigns that used different instruments, and covers only five materials.
- No model was trained on 1 g data. The models are valid only inside the tested microgravity conditions, and the domain check flags everything else.

**NASA data (all CC0, NASA Physical Sciences Informatics, https://psi.nasa.gov)**
- BASS-II flame spread: NASA PSI-25; Link et al. 2018, *Scientific Reports* 8:120, doi:10.1038/s41598-017-18398-4 (Table S1)
- FLEX droplet combustion: NASA PSI-69, DOI 10.60555/mbq8-0451
- SPICE smoke points: NASA PSI-107, DOI 10.60555/9kbh-e962
- SAME / SAME-R smoke particles: NASA PSI-101, PSI-102

**Tools:** Python, pandas, NumPy, scikit-learn, matplotlib, Google Colab.

**Use of AI.** As the Space Apps Project Submission Guide requires, this notebook states that AI assistance was used. Claude (Anthropic) assisted in writing parts of the code and documentation for the data pipeline and the model-selection workflow. The team member named above directed the work and checked the results: every data value is traced to its NASA source, and every reported score is reproduced by this notebook. No AI-generated images are used. The figures are plotted directly from the NASA data.

## 11. Automated tests

The test suite checks five things:
- the data contract, including corrupted copies that must be rejected
- reproducible ETL
- leakage guards for groups and targets
- artifact integrity
- the prediction contract, exact Shapley, out-of-domain handling and physics behaviour

In [ ]:
!python ../tests/run_tests.py